Create a  Weather Tool MCP Server that any AI agent can use with sample use case

In [1]:
!pip install "mcp[cli]" httpx

In [4]:
!pip install -q gradio httpx

import asyncio
from typing import Any
import httpx
import gradio as gr

# ============================================
# CONFIG
# ============================================

GEOCODE_API = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_API = "https://api.open-meteo.com/v1/forecast"


# ============================================
# HELPER FUNCTION (ASYNC API CALL)
# ============================================

async def make_request(url: str, params: dict[str, Any]) -> dict[str, Any] | None:
    """
    Sends HTTP request to external API and returns JSON response.
    Handles errors safely.
    """
    headers = {
        "User-Agent": "weather-gradio-app/1.0"
    }

    async with httpx.AsyncClient(timeout=20.0, headers=headers) as client:
        try:
            response = await client.get(url, params=params)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            return {"error": f"Request failed: {e}"}


# ============================================
# WEATHER FUNCTIONS
# ============================================

async def get_current_weather(location: str) -> str:
    """
    Returns current weather details for a given location.
    """
    geo_data = await make_request(
        GEOCODE_API,
        {"name": location, "count": 1, "language": "en", "format": "json"},
    )

    if not geo_data or geo_data.get("error"):
        return f"Error: {geo_data.get('error', 'Unable to fetch location data')}"

    if not geo_data.get("results"):
        return f"Location not found: {location}"

    place = geo_data["results"][0]
    latitude = place["latitude"]
    longitude = place["longitude"]
    resolved_name = f"{place['name']}, {place.get('country', 'Unknown')}"

    weather_data = await make_request(
        FORECAST_API,
        {
            "latitude": latitude,
            "longitude": longitude,
            "current": [
                "temperature_2m",
                "relative_humidity_2m",
                "apparent_temperature",
                "is_day",
                "precipitation",
                "weather_code",
                "wind_speed_10m",
            ],
            "timezone": "auto",
        },
    )

    if not weather_data or weather_data.get("error"):
        return f"Error: {weather_data.get('error', 'Unable to fetch weather data')}"

    if "current" not in weather_data:
        return f"Weather data unavailable for {resolved_name}"

    current = weather_data["current"]

    return (
        f"Current weather for {resolved_name}\n\n"
        f"Temperature: {current.get('temperature_2m')}°C\n"
        f"Feels like: {current.get('apparent_temperature')}°C\n"
        f"Humidity: {current.get('relative_humidity_2m')}%\n"
        f"Wind speed: {current.get('wind_speed_10m')} km/h\n"
        f"Precipitation: {current.get('precipitation')} mm\n"
        f"Weather code: {current.get('weather_code')}\n"
        f"Daytime: {'Yes' if current.get('is_day') == 1 else 'No'}"
    )


async def get_weather_forecast(location: str, days: int = 3) -> str:
    """
    Returns weather forecast (1–7 days).
    """
    days = max(1, min(days, 7))

    geo_data = await make_request(
        GEOCODE_API,
        {"name": location, "count": 1, "language": "en", "format": "json"},
    )

    if not geo_data or geo_data.get("error"):
        return f"Error: {geo_data.get('error', 'Unable to fetch location data')}"

    if not geo_data.get("results"):
        return f"Location not found: {location}"

    place = geo_data["results"][0]
    latitude = place["latitude"]
    longitude = place["longitude"]
    resolved_name = f"{place['name']}, {place.get('country', 'Unknown')}"

    forecast_data = await make_request(
        FORECAST_API,
        {
            "latitude": latitude,
            "longitude": longitude,
            "daily": [
                "weather_code",
                "temperature_2m_max",
                "temperature_2m_min",
                "precipitation_sum",
            ],
            "forecast_days": days,
            "timezone": "auto",
        },
    )

    if not forecast_data or forecast_data.get("error"):
        return f"Error: {forecast_data.get('error', 'Unable to fetch forecast data')}"

    if "daily" not in forecast_data:
        return f"Forecast unavailable for {resolved_name}"

    daily = forecast_data["daily"]
    lines = [f"{days}-day forecast for {resolved_name}\n"]

    for i in range(len(daily["time"])):
        lines.append(
            f"{daily['time'][i]}:\n"
            f"  Min Temp: {daily['temperature_2m_min'][i]}°C\n"
            f"  Max Temp: {daily['temperature_2m_max'][i]}°C\n"
            f"  Rain: {daily['precipitation_sum'][i]} mm\n"
            f"  Weather Code: {daily['weather_code'][i]}\n"
        )

    return "\n".join(lines)


# ============================================
# SYNC WRAPPERS FOR GRADIO
# ============================================

def current_weather_ui(location):
    return asyncio.run(get_current_weather(location))

def forecast_ui(location, days):
    return asyncio.run(get_weather_forecast(location, int(days)))


# ============================================
# GRADIO UI
# ============================================

with gr.Blocks() as demo:
    gr.Markdown("# Weather Tool Demo")
    gr.Markdown("Notebook-safe UI version of your Weather Tool.")

    with gr.Tab("Current Weather"):
        loc1 = gr.Textbox(label="Location", placeholder="Enter city name, e.g. Delhi")
        out1 = gr.Textbox(label="Output", lines=10)
        btn1 = gr.Button("Get Current Weather")
        btn1.click(fn=current_weather_ui, inputs=loc1, outputs=out1)

    with gr.Tab("Forecast"):
        loc2 = gr.Textbox(label="Location", placeholder="Enter city name, e.g. Mumbai")
        days = gr.Slider(1, 7, value=3, step=1, label="Forecast Days")
        out2 = gr.Textbox(label="Output", lines=14)
        btn2 = gr.Button("Get Forecast")
        btn2.click(fn=forecast_ui, inputs=[loc2, days], outputs=out2)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4a0ffba136e71e1805.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
